## News Scraping from Malaysian News

## Install and import required libraries

In [23]:
!pip install newspaper3k
!pip install autoscraper
!pip install requests


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import pandas as pd
from autoscraper import AutoScraper
from bs4 import BeautifulSoup
import requests

In [51]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time
import re
import csv

BASE = "https://www.freemalaysiatoday.com"
LIST_BASE = BASE + "/category/category/business/local-business"
API_URL = BASE + "/api/more-vertical-posts"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Regex date extractor from URL
date_pattern = re.compile(r"/(\d{4})/(\d{2})/(\d{2})/")

def get_date_from_url(url):
    m = date_pattern.search(url)
    if not m:
        return None
    y, mth, d = map(int, m.groups())
    return datetime(y, mth, d).date()

# Scrape full article text
def scrape_article(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
    except requests.HTTPError as e:
        print(f"Skipping {url} ({e})")
        return None
    except requests.RequestException as e:
        # handles timeouts, connection errors, etc.
        print(f"Request error for {url}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    paragraphs = [p.get_text(strip=True) for p in soup.select("p")]
    body = "\n".join(p for p in paragraphs if p)

    art_date = get_date_from_url(url)

    return {
        "url": url,
        "title": title,
        "date": art_date.isoformat() if art_date else "",
        "body": body,
    }

# Extract links from initial HTML page
def get_initial_links():
    resp = requests.get(LIST_BASE, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    links = []
    for a in soup.find_all("a"):
        href = a.get("href") or ""
        if "/category/business/" in href and date_pattern.search(href):
            if href.startswith("/"):
                href = BASE + href
            links.append(href)

    links = sorted(set(links))
    print("HTML page 1:", len(links), "links")
    return links

# API pagination using offset
import requests

from datetime import datetime, timedelta

def parse_api_date(dt_str: str):
    if not dt_str:
        return None
    # API format looks like '2025-11-17T18:10:18'
    try:
        return datetime.fromisoformat(dt_str)
    except Exception:
        return None

def fetch_api_links(start_offset=25, step=20, cutoff_date=None):
    """
    Paginate /api/more-vertical-posts until:
      - no edges, OR
      - oldest article in a batch is older than cutoff_date (if provided)
    """
    all_links = set()
    offset = start_offset

    while True:
        payload = {
            "categorySlug": "local-business",
            "offset": offset
        }

        try:
            resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=30)
            resp.raise_for_status()
        except requests.exceptions.ReadTimeout:
            print(f"Timeout at offset {offset}, stopping API pagination.")
            break
        except requests.RequestException as e:
            print(f"Request error at offset {offset}: {e}, stopping API pagination.")
            break

        data = resp.json()
        posts = data.get("posts", {})
        edges = posts.get("edges", [])

        print(f"API offset {offset}: {len(edges)} edges")

        if not edges:
            print(f"API offset {offset}: no edges, stopping")
            break

        links_this_offset = 0
        oldest_in_batch = None

        for edge in edges:
            node = edge.get("node", {})
            uri = node.get("uri") or node.get("link") or ""
            date_str = node.get("date")  # e.g. '2025-11-17T18:10:18'
            dt = parse_api_date(date_str)

            if dt and (oldest_in_batch is None or dt < oldest_in_batch):
                oldest_in_batch = dt

            if not uri:
                continue

            url = uri
            if url.startswith("/"):
                url = BASE + url

            if "/category/business/" in url and date_pattern.search(url):
                if url not in all_links:
                    all_links.add(url)
                    links_this_offset += 1

        print(f"API offset {offset}: {links_this_offset} new links")
        print(f"Total unique API links so far: {len(all_links)}")

        # stop if this batch is already older than our cutoff
        if cutoff_date is not None and oldest_in_batch is not None:
            if oldest_in_batch.date() < cutoff_date:
                print(f"Oldest in batch {oldest_in_batch.date()} < cutoff {cutoff_date}, stopping.")
                break

        offset += step
        time.sleep(1)

    return sorted(all_links)





import json

def debug_one_offset(offset=25):
    payload = {
        "categorySlug": "local-business",
        "offset": offset
    }
    resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=10)
    print("Status:", resp.status_code)
    data = resp.json()
    print("Top-level keys:", data.keys())

    posts = data.get("posts", {})
    print("type(posts):", type(posts), "| len(posts):", len(posts))

    if not posts:
        return

    # posts is a dict – inspect its keys and a snippet of each value
    for k, v in posts.items():
        print(f"Key in posts: {k} | type: {type(v)}")
        if isinstance(v, str):
            print("Value snippet:", repr(v)[:400])
        elif isinstance(v, dict):
            print("Value keys:", v.keys())
            print("Value snippet:", json.dumps(v, indent=2)[:400])
        else:
            print("Value repr:", repr(v)[:400])




# MAIN
if __name__ == "__main__":
    today = datetime.today().date()
    # ~6 months ≈ 180 days
    cutoff = today - timedelta(days=180)
    print("Today:", today, "| Cutoff:", cutoff)

    all_links = set()

    # 1) Latest few days from HTML
    all_links.update(get_initial_links())

    # 2) Older stuff via API, stopping when oldest batch < cutoff
    all_links.update(fetch_api_links(cutoff_date=cutoff))

    print("Total candidate links:", len(all_links))

    articles = []
    for url in sorted(all_links):
        art_date = get_date_from_url(url)
        if not art_date:
            continue
        if art_date < cutoff:
            continue

        art = scrape_article(url)
        if art is None:   # in case of 404 / timeout
            continue
        articles.append(art)
        print(f"[{art_date}] {art['title']}")
        time.sleep(1)

    print("\nTotal Local Business articles (last ~6 months):", len(articles))

    with open("local_business_last_6_months.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["url", "title", "date", "body"])
        writer.writeheader()
        writer.writerows(articles)

    print("Saved to local_business_last_6_months.csv")



Today: 2025-11-20 | Cutoff: 2025-05-24
HTML page 1: 16 links
API offset 25: 20 edges
API offset 25: 12 new links
Total unique API links so far: 12
API offset 45: 20 edges
API offset 45: 12 new links
Total unique API links so far: 24
API offset 65: 20 edges
API offset 65: 14 new links
Total unique API links so far: 38
API offset 85: 20 edges
API offset 85: 13 new links
Total unique API links so far: 51
API offset 105: 20 edges
API offset 105: 15 new links
Total unique API links so far: 66
API offset 125: 20 edges
API offset 125: 13 new links
Total unique API links so far: 79
API offset 145: 20 edges
API offset 145: 13 new links
Total unique API links so far: 92
API offset 165: 20 edges
API offset 165: 10 new links
Total unique API links so far: 102
API offset 185: 20 edges
API offset 185: 15 new links
Total unique API links so far: 117
API offset 205: 20 edges
API offset 205: 11 new links
Total unique API links so far: 128
API offset 225: 20 edges
API offset 225: 14 new links
Total uniq